In [ ]:
# Installations
!pip install langchain langchain-community langchain-huggingface langchain-chroma langchain-groq sentence-transformers chromadb pymupdf pypdf python-dotenv tqdm networkx matplotlib

In [ ]:
import os
import json
import networkx as nx
import matplotlib.pyplot as plt

from dotenv import load_dotenv

from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document

In [ ]:
load_dotenv()
GROQ_API_KEY = os.environ['GROQ_API_KEY']
if GROQ_API_KEY is None:
  raise ValueError("GROQ_API_KEY is not set")

In [ ]:
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

llm = ChatGroq(
    model="meta-llama/llama-4-scout-17b-16e-instruct",
    temperature=0,
    api_key=GROQ_API_KEY
)

In [ ]:
# Data Path of PDFs used
DATA_PATH = "/content/drive/MyDrive/RAG_Article/data"

In [ ]:
# Load the documents
documents = []

for file in os.listdir(DATA_PATH):
    if file.endswith(".pdf"):
        loader = PyMuPDFLoader(os.path.join(DATA_PATH, file))
        documents.extend(loader.load())

print(f"Loaded {len(documents)} pages.")

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

print(len(chunks))

In [ ]:
graph_prompt = ChatPromptTemplate.from_template("""
You are an expert at constructing knowledge graphs from technical documentation.

Extract:

1. Important technical entities.
2. A concise description for each entity.
3. Relationships between entities.

Return ONLY valid JSON.

The JSON must follow exactly this schema:

{{
    "entities":[
        {{
            "name":"Entity Name",
            "description":"Short description of the entity."
        }}
    ],
    "relationships":[
        {{
            "source":"Entity A",
            "relation":"uses",
            "target":"Entity B"
        }}
    ]
}}

Rules:

- Include frameworks, algorithms, techniques, models, datasets, APIs, tools and libraries.
- Description should be 1 sentence.
- Use the relation exactly as implied by the text.
- Don't hallucinate entities.
- Don't return markdown.
- Return ONLY JSON.

TEXT:

{text}
""")

In [ ]:
def extract_graph(chunk):

    prompt = graph_prompt.format(
        text=chunk.page_content
    )

    response = llm.invoke(prompt)

    return response.content

In [ ]:
import json

def parse_graph(llm_output):

    llm_output = llm_output.strip()

    if llm_output.startswith("```"):
        llm_output = llm_output.replace("```json", "")
        llm_output = llm_output.replace("```", "")
        llm_output = llm_output.strip()

    try:
        return json.loads(llm_output)

    except Exception as e:
        print(e)
        return None

In [ ]:
graph_db = nx.Graph()

In [ ]:
for chunk_id, chunk in enumerate(chunks):

    print(f"Processing chunk {chunk_id + 1}/{len(chunks)}")

    graph_json = extract_graph(chunk)
    graph_json = parse_graph(graph_json)

    if graph_json is None:
        continue

    entities = graph_json.get("entities", [])
    relations = graph_json.get("relationships", [])

    # Add nodes
    for entity in entities:

      name = entity["name"]
      description = entity["description"]

      if not graph_db.has_node(name):

          graph_db.add_node(
              name,
              description=description,
              chunks=[]
          )

      graph_db.nodes[name]["chunks"].append(chunk_id)

    # Add edges
    for rel in relations:

        source = rel["source"]
        target = rel["target"]
        relation = rel["relation"]

        graph_db.add_edge(
            source,
            target,
            relation=relation
        )

In [ ]:
nodes = list(graph_db.nodes(data=True))
for node in nodes:
    print(node[0])

In [ ]:
entity_documents = []

nodes = list(graph_db.nodes(data=True))
for node in nodes:

    description = node[1].get("description")

    chunk_ids = node[1].get("chunks")

    entity_documents.append(

        Document(

            page_content=description if description is not None else "",

            metadata={
                "entity": node[0],
                "chunks": chunk_ids
            }

        )

    )

print(len(entity_documents))

In [ ]:
entity_vector_db = Chroma.from_documents(
    documents=entity_documents,
    embedding=embedding_model,
    persist_directory="entity_db"
)

In [ ]:
entity_retriever = entity_vector_db.as_retriever(
    search_kwargs={"k":3}
)

In [ ]:
def find_entities(query):

    query = query.lower()

    matched = []

    for node in graph_db.nodes():

        if node.lower() in query:
            matched.append(node)

    return matched

In [ ]:
query = "List the different opeartions on arrays"

entities = entity_retriever.invoke(query)

print(entities)

In [ ]:
retrieved_entities = [

    doc.metadata["entity"]

    for doc in entities

]

print(retrieved_entities)

In [ ]:
def traverse_graph(entities):

    connected = set()

    for entity in entities:

        connected.add(entity)

        neighbors = graph_db.neighbors(entity)

        for neighbor in neighbors:
            connected.add(neighbor)

    return list(connected)

In [ ]:
related_entities = traverse_graph(retrieved_entities)

print(related_entities)

In [ ]:
def retrieve_chunk_ids(entities):

    chunk_ids = set()

    for entity in entities:

        chunk_ids.update(
            graph_db.nodes[entity]["chunks"]
        )

    return sorted(chunk_ids)

In [ ]:
chunk_ids = retrieve_chunk_ids(
    related_entities
)

print(chunk_ids)

In [ ]:
retrieved_docs = [
    chunks[idx]
    for idx in chunk_ids
]

In [ ]:
context = "\n\n".join(
    doc.page_content
    for doc in retrieved_docs
)

In [ ]:
prompt = ChatPromptTemplate.from_template("""
You are a helpful AI assistant.

Answer the question ONLY using the provided context.

If the answer isn't contained in the context, say so.

Context:

{context}

Question:

{question}
""")

In [ ]:
formatted_prompt = prompt.format(
    context=context,
    question=query
)

response = llm.invoke(formatted_prompt)

print(response.content)

In [ ]:
!jupyter nbconvert --ClearMetadataPreprocessor.enabled=True --inplace Graph_RAG.ipynb

[NbConvertApp] WARNING | pattern 'Graph_RAG.ipynb' matched no files
This application is used to convert notebook files (*.ipynb)
        to various other formats.


Options
The options below are convenience aliases to configurable class-options,
as listed in the "Equivalent to" description-line of the aliases.
To see all configurable class-options for some <cmd>, use:
    <cmd> --help-all

--debug
    set log level to logging.DEBUG (maximize logging output)
    Equivalent to: [--Application.log_level=10]
--show-config
    Show the application's configuration (human-readable format)
    Equivalent to: [--Application.show_config=True]
--show-config-json
    Show the application's configuration (json format)
    Equivalent to: [--Application.show_config_json=True]
--generate-config
    generate default config file
    Equivalent to: [--JupyterApp.generate_config=True]
-y
    Answer yes to any questions instead of prompting.
    Equivalent to: [--JupyterApp.answer_yes=True]
--execute
    E